In [2]:
import pandas as pd


In [4]:
df=pd.read_csv("../data/raw/train.csv")

In [5]:
df["FamilySize"]=df["SibSp"]+df["Parch"]+1

In [6]:
pd.crosstab(df["FamilySize"], df["Survived"], normalize="index")

Survived,0,1
FamilySize,,
1,0.696462,0.303538
2,0.447205,0.552795
3,0.421569,0.578431
4,0.275862,0.724138
5,0.800000,0.200000
6,0.863636,0.136364
7,0.666667,0.333333
8,1.000000,0.000000
11,1.000000,0.000000


In [7]:
df["isAlone"]=(df["FamilySize"]==1).astype(int)

In [8]:
pd.crosstab(df["isAlone"], df["Survived"], normalize="index")

Survived,0,1
isAlone,,
0,0.494350,0.505650
1,0.696462,0.303538


In [9]:
pd.crosstab([df["isAlone"],df["Sex"]], df["Survived"], normalize="index")

Survived               0         1
isAlone Sex                       
0       female  0.287234  0.712766
        male    0.728916  0.271084
1       female  0.214286  0.785714
        male    0.844282  0.155718

In [14]:
df["FamilySizeGroup"] = pd.cut(
    df["FamilySize"],
    bins=[0, 1, 4, 7, 20],
    labels=["Alone", "Small", "Medium", "Large"]
)

In [15]:
pd.crosstab(
    df["FamilySizeGroup"],
    df["Survived"],
    normalize="index"
)

Survived,0,1
FamilySizeGroup,,
Alone,0.696462,0.303538
Small,0.421233,0.578767
Medium,0.795918,0.204082
Large,1.000000,0.000000


In [16]:
ticket_counts=df["Ticket"].value_counts()
ticket_counts.head(20)

Ticket
347082          7
1601            7
CA. 2343        7
3101295         6
CA 2144         6
347088          6
382652          5
S.O.C. 14879    5
349909          4
347077          4
19950           4
W./C. 6608      4
4133            4
LINE            4
113781          4
17421           4
PC 17757        4
113760          4
2666            4
347742          3
Name: count, dtype: int64

In [17]:
df['TicketGroupSize']=df["Ticket"].map(ticket_counts)

In [18]:
pd.crosstab(df["TicketGroupSize"], df["Survived"], normalize="index")

Survived,0,1
TicketGroupSize,,
1,0.702011,0.297989
2,0.425532,0.574468
3,0.301587,0.698413
4,0.500000,0.500000
5,1.000000,0.000000
6,1.000000,0.000000
7,0.761905,0.238095


In [20]:
ticket_summary=df.groupby("Ticket").agg(
    TicketGroupSize=("PassengerId","count"),
    SurvivedRate=("Survived","mean")
)
ticket_summary.sort_values("TicketGroupSize",ascending=False).head(20)

,TicketGroupSize,SurvivedRate
Ticket,,
1601,7,0.714286
CA. 2343,7,0.000000
347082,7,0.000000
CA 2144,6,0.000000
3101295,6,0.000000
347088,6,0.000000
382652,5,0.000000
S.O.C. 14879,5,0.000000
349909,4,0.000000


In [21]:
df[df["Ticket"].isin(
    ticket_summary[ticket_summary["TicketGroupSize"] >= 5].index
)].sort_values("Ticket")

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,isAlone,FamilySizeGroup,TicketGroupSize
74,75,1,3,"Bing, Mr. Lee",male,32.0,0,0,1601,56.4958,NaN,S,1,1,Alone,7
169,170,0,3,"Ling, Mr. Lee",male,28.0,0,0,1601,56.4958,NaN,S,1,1,Alone,7
509,510,1,3,"Lang, Mr. Fang",male,26.0,0,0,1601,56.4958,NaN,S,1,1,Alone,7
643,644,1,3,"Foo, Mr. Choong",male,NaN,0,0,1601,56.4958,NaN,S,1,1,Alone,7
692,693,1,3,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,NaN,S,1,1,Alone,7
826,827,0,3,"Lam, Mr. Len",male,NaN,0,0,1601,56.4958,NaN,S,1,1,Alone,7
838,839,1,3,"Chip, Mr. Chang",male,32.0,0,0,1601,56.4958,NaN,S,1,1,Alone,7
50,51,0,3,"Panula, Master. Juha Niilo",male,7.0,4,1,3101295,39.6875,NaN,S,6,0,Medium,6
164,165,0,3,"Panula, Master. Eino Viljami",male,1.0,4,1,3101295,39.6875,NaN,S,6,0,Medium,6
266,267,0,3,"Panula, Mr. Ernesti Arvid",male,16.0,4,1,3101295,39.6875,NaN,S,6,0,Medium,6


In [68]:
features=["Pclass","Sex","Age","Fare","Embarked","FamilySize","isAlone"]

In [69]:
from sklearn.model_selection import train_test_split
X=df.drop("Survived",axis=1)
y=df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)



In [70]:
X_train=X_train[features]
X_test=X_test[features]

In [71]:
numeric_features = [
    "Pclass",
    "Age",
    "Fare",
    "FamilySize",
    "isAlone"
]

categorical_features = [
    "Sex",
    "Embarked"
]

In [72]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model_fe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [73]:
model_fe.fit(X_train, y_train)

y_pred = model_fe.predict(X_test)

In [74]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy_fe = accuracy_score(y_test, y_pred)

print("Feature Engineered Accuracy:", accuracy_fe)
print()
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Feature Engineered Accuracy: 0.7988826815642458

[[95 15]
 [21 48]]

              precision    recall  f1-score   support

           0       0.82      0.86      0.84       110
           1       0.76      0.70      0.73        69

    accuracy                           0.80       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.80      0.80      0.80       179



In [75]:
df["TicketGroupSize"] = df["Ticket"].map(
    df["Ticket"].value_counts()
)

df["FarePerPerson"] = df["Fare"] / df["TicketGroupSize"]

In [76]:
df.groupby("Survived")["FarePerPerson"].mean()

Survived
0    13.523869
1    24.635628
Name: FarePerPerson, dtype: float64

In [98]:
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 30, 50, 80],
    labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]
)

In [134]:
df["Sex_pclass"] = df["Sex"] + "_" + df["Pclass"].astype(str)

In [155]:
df["HasCabin"] = df["Cabin"].notna().astype(int)

pd.crosstab(
    df["HasCabin"],
    df["Survived"],
    normalize="index"
)

Survived,0,1
HasCabin,,
0,0.700146,0.299854
1,0.333333,0.666667


In [ ]:
features1=["Pclass","Sex","AgeGroup","Fare","Embarked","FamilySize","isAlone"]

In [157]:
from sklearn.model_selection import train_test_split
X=df.drop("Survived",axis=1)
y=df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)



In [158]:
X_train1=X_train[features1]
X_test1=X_test[features1]

In [ ]:
numeric_features = [
    "Pclass",
    "Fare",
    "FamilySize",
    "isAlone"
]

categorical_features = [
    "Sex",
    "AgeGroup",
    "Embarked"
]

In [160]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model_1 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [161]:
model_1.fit(X_train1, y_train)

y_pred1 = model_1.predict(X_test1)

In [162]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy_1 = accuracy_score(y_test, y_pred1)

print("Feature Engineered Accuracy:", accuracy_1)
print()
print(confusion_matrix(y_test, y_pred1))
print()
print(classification_report(y_test, y_pred1))

Feature Engineered Accuracy: 0.7821229050279329

[[90 20]
 [19 50]]

              precision    recall  f1-score   support

           0       0.83      0.82      0.82       110
           1       0.71      0.72      0.72        69

    accuracy                           0.78       179
   macro avg       0.77      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179



In [97]:
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 30, 50, 80],
    labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]
)

In [165]:
from sklearn.ensemble import GradientBoostingClassifier
rf_model=Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        random_state=42
    ))
])
rf_model.fit(X_train1, y_train)
y_pred_rf = rf_model.predict(X_test1)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.7877094972067039
[[94 16]
 [22 47]]
              precision    recall  f1-score   support

           0       0.81      0.85      0.83       110
           1       0.75      0.68      0.71        69

    accuracy                           0.79       179
   macro avg       0.78      0.77      0.77       179
weighted avg       0.79      0.79      0.79       179



| Model                  |   Accuracy |
| ---------------------- | ---------: |
| 🥇 Logistic Regression | **81.56%** |
| Random Forest          |     78.77% |
| Gradient Boosting      |     78.77% |


In [166]:
df["Deck"]=df['Cabin'].str[0].fillna('U')
print(df['Deck'].value_counts())

Deck
U    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64


In [167]:
pd.crosstab(df["Deck"], df["Survived"], normalize="index")

Survived,0,1
Deck,,
A,0.533333,0.466667
B,0.255319,0.744681
C,0.406780,0.593220
D,0.242424,0.757576
E,0.250000,0.750000
F,0.384615,0.615385
G,0.500000,0.500000
T,1.000000,0.000000
U,0.700146,0.299854


In [183]:
df['title']=df['Name'].str.extract(r",\s*([^,\.]+)\.")[0].str.strip()
print(df['title'].value_counts())

title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64


In [185]:
title_counts = df["title"].value_counts()

df["title"] = df["title"].where(
    df["title"].isin(["Mr", "Miss", "Mrs", "Master"]),
    "Rare"
)

In [195]:
features_df=[
    "Pclass",
    "Sex",
    "AgeGroup",
    "Fare",
    "Embarked",
    "FamilySize",
    "isAlone",
    
    'title'
]

In [197]:
numeric_features = [
    "Pclass",
    "Fare",
    "FamilySize",
    "isAlone"]
categorical_features = [
    "Sex",
    "AgeGroup",
    "Embarked",
    
    'title'
]

In [198]:
from sklearn.model_selection import train_test_split
X=df.drop("Survived",axis=1)
y=df["Survived"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


In [199]:
X_train=X_train[features_df]
X_test=X_test[features_df]

In [200]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model_df= Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [201]:
model_df.fit(X_train,y_train)
pred_y1=model_df.predict(X_test)

In [202]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy_1 = accuracy_score(y_test, pred_y1)

print("Feature Engineered Accuracy:", accuracy_1)
print()
print(confusion_matrix(y_test, pred_y1))
print()
print(classification_report(y_test, pred_y1))

Feature Engineered Accuracy: 0.8324022346368715

[[98 12]
 [18 51]]

              precision    recall  f1-score   support

           0       0.84      0.89      0.87       110
           1       0.81      0.74      0.77        69

    accuracy                           0.83       179
   macro avg       0.83      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179



In [203]:
import joblib
joblib.dump(
    model_df,
    "../Model/titanic_logistic_best_82_68.pkl"
)

['../Model/titanic_logistic_best_82_68.pkl']

Contribution of each feature till now

In [204]:
model_df.named_steps

{'preprocessor': ColumnTransformer(transformers=[('num',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='median'))]),
                                  ['Pclass', 'Fare', 'FamilySize', 'isAlone']),
                                 ('cat',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='most_frequent')),
                                                  ('encoder',
                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                  ['Sex', 'AgeGroup', 'Embarked', 'title'])]),
 'classifier': LogisticRegression(max_iter=1000)}

In [205]:
classifier=model_df.named_steps["classifier"]

In [206]:
feature_name=model_df.named_steps["preprocessor"].get_feature_names_out()

In [207]:
coefficients = classifier.coef_[0]

In [208]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": feature_name,
    "Coefficient": coefficients
})

feature_importance = feature_importance.sort_values(
    "Coefficient",
    ascending=False
)

print(feature_importance)

                     Feature  Coefficient
14         cat__title_Master     1.370608
4            cat__Sex_female     0.940439
7        cat__AgeGroup_Child     0.871777
17            cat__title_Mrs     0.390096
12           cat__Embarked_Q     0.289997
6        cat__AgeGroup_Adult     0.069666
11           cat__Embarked_C     0.055932
1                  num__Fare     0.003689
9         cat__AgeGroup_Teen    -0.016142
10  cat__AgeGroup_YoungAdult    -0.089628
15           cat__title_Miss    -0.157232
13           cat__Embarked_S    -0.322395
3               num__isAlone    -0.364069
2            num__FamilySize    -0.468675
18           cat__title_Rare    -0.473192
8       cat__AgeGroup_Senior    -0.812139
0                num__Pclass    -0.900172
5              cat__Sex_male    -0.916904
16             cat__title_Mr    -1.106746


| Experiment                              | Accuracy | Decision            |
| --------------------------------------- | -------: | ------------------- |
| Logistic Regression + original features | Baseline | Reference           |
| + FamilySize + IsAlone                  |   80.45% | Keep                |
| + AgeGroup                              |   81.56% | Keep                |
| + Title                                 |     ~83% | 🏆 Current best     |
| Random Forest                           |   78.77% | Not competitive yet |
| Gradient Boosting                       |   78.77% | Not competitive yet |


In [209]:
features = [
    "Pclass",
    "Sex",
    "AgeGroup",
    "Fare",
    "Embarked",
    "FamilySize",
    "IsAlone",
    "Title"
]

In [210]:
%pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [212]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)



In [219]:
xgb_model=Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])

In [220]:
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

Accuracy: 0.8212290502793296
[[97 13]
 [19 50]]
              precision    recall  f1-score   support

           0       0.84      0.88      0.86       110
           1       0.79      0.72      0.76        69

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179



| Model                          |   Accuracy |
| ------------------------------ | ---------: |
| 🥇 Logistic Regression + Title |   **~83%** |
| XGBoost + same features        | **82.12%** |
| Random Forest                  |     78.77% |
| Gradient Boosting              |     78.77% |


In [223]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    xgb_model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

print("CV scores:", scores)
print("Mean CV accuracy:", scores.mean())
print("Std:", scores.std())

CV scores: [0.83216783 0.7972028  0.83802817 0.85915493 0.83098592]
Mean CV accuracy: 0.8315079286910274
Std: 0.019921948743471404


| Model               |    Mean CV |       Std |
| ------------------- | ---------: | --------: |
| Logistic Regression |     81.46% |     2.30% |
| **XGBoost**         | **83.15%** | **1.99%** |


In [225]:
for depth in [2, 3, 4, 5, 6]:

    tuned_xgb = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=200,
            max_depth=depth,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ])

    scores = cross_val_score(
        tuned_xgb,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    print(
        f"depth={depth} "
        f"Mean={scores.mean():.4f} "
        f"Std={scores.std():.4f}"
    )

depth=2 Mean=0.8202 Std=0.0120
depth=3 Mean=0.8315 Std=0.0199
depth=4 Mean=0.8357 Std=0.0177
depth=5 Mean=0.8399 Std=0.0201
depth=6 Mean=0.8413 Std=0.0149


In [227]:
for lr in [0.01, 0.03, 0.05, 0.1, 0.2]:

    tuned_xgb = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=lr,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ])

    scores = cross_val_score(
        tuned_xgb,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    print(
        f"lr={lr:<4} "
        f"Mean={scores.mean():.4f} "
        f"Std={scores.std():.4f}"
    )

lr=0.01 Mean=0.8287 Std=0.0203
lr=0.03 Mean=0.8385 Std=0.0180
lr=0.05 Mean=0.8413 Std=0.0149
lr=0.1  Mean=0.8427 Std=0.0140
lr=0.2  Mean=0.8357 Std=0.0126


XGBoost: max_depth=6, learning_rate=0.1
84.27% ± 1.40%
#Current Best

In [229]:
for n in [50, 100, 200, 300, 500]:

    tuned_xgb = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=n,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ])

    scores = cross_val_score(
        tuned_xgb,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    print(
        f"n_estimators={n:<3} "
        f"Mean={scores.mean():.4f} "
        f"Std={scores.std():.4f}"
    )

n_estimators=50  Mean=0.8427 Std=0.0243
n_estimators=100 Mean=0.8343 Std=0.0154
n_estimators=200 Mean=0.8427 Std=0.0140
n_estimators=300 Mean=0.8329 Std=0.0098
n_estimators=500 Mean=0.8216 Std=0.0197


In [230]:
for subsample in [0.6, 0.7, 0.8, 0.9, 1.0]:

    tuned_xgb = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=subsample,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ])

    scores = cross_val_score(
        tuned_xgb,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    print(
        f"subsample={subsample:<3} "
        f"Mean={scores.mean():.4f} "
        f"Std={scores.std():.4f}"
    )

subsample=0.6 Mean=0.8343 Std=0.0127
subsample=0.7 Mean=0.8287 Std=0.0164
subsample=0.8 Mean=0.8427 Std=0.0140
subsample=0.9 Mean=0.8399 Std=0.0153
subsample=1.0 Mean=0.8343 Std=0.0128


In [232]:
for colsample in [0.6, 0.7, 0.8, 0.9, 1.0]:

    tuned_xgb = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=colsample,
            eval_metric="logloss",
            random_state=42
        ))
    ])

    scores = cross_val_score(
        tuned_xgb,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    print(
        f"colsample={colsample:<3} "
        f"Mean={scores.mean():.4f} "
        f"Std={scores.std():.4f}"
    )

colsample=0.6 Mean=0.8357 Std=0.0166
colsample=0.7 Mean=0.8399 Std=0.0132
colsample=0.8 Mean=0.8427 Std=0.0140
colsample=0.9 Mean=0.8413 Std=0.0146
colsample=1.0 Mean=0.8343 Std=0.0127


In [233]:
df["TicketGroupSize"] = df["Ticket"].map(df["Ticket"].value_counts())

In [239]:
features = [
    "Pclass",
    "Sex",
    "AgeGroup",
    "Fare",
    "Embarked",
    "FamilySize",
    "isAlone",
    "title"
]

In [240]:
numeric_features = [
    "Pclass",
    "Fare",
    "FamilySize",
    "isAlone"
]
X_fe = df[features]
y = df["Survived"]


# 4. Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


# 5. Current best XGBoost configuration
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])


# 6. Same 5-fold CV
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# 7. Evaluate
scores = cross_val_score(
    xgb_model,
    X_fe,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Fold scores:", scores)
print("Mean accuracy:", scores.mean())
print("Std:", scores.std())

Fold scores: [0.83798883 0.85955056 0.81460674 0.83707865 0.84269663]
Mean accuracy: 0.8383842822170611
Std: 0.014379431303383773


Current benchmark remains:

XGBoost: 84.27% ± 1.40%

In [237]:
print("Features:")
print(features)

print("\nNumeric:")
print(numeric_features)

print("\nCategorical:")
print(categorical_features)

print("\nX shape:")
print(X_fe.shape)

Features:
['Pclass', 'Sex', 'AgeGroup', 'Fare', 'Embarked', 'FamilySize', 'isAlone', 'title', 'TicketGroupSize']

Numeric:
['Pclass', 'Fare', 'FamilySize', 'isAlone', 'TicketGroupSize']

Categorical:
['Sex', 'AgeGroup', 'Embarked', 'title']

X shape:
(891, 9)


In [238]:
print(X_fe.columns.tolist())

['Pclass', 'Sex', 'AgeGroup', 'Fare', 'Embarked', 'FamilySize', 'isAlone', 'title', 'TicketGroupSize']


In [245]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Train current best model
xgb_model.fit(X_train[features], y_train)

# Predictions
y_pred = xgb_model.predict(X_test[features])

# Accuracy
print("Test Accuracy:", accuracy_score(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Test Accuracy: 0.7877094972067039

Confusion Matrix:
[[92 18]
 [20 49]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.84      0.83       110
           1       0.73      0.71      0.72        69

    accuracy                           0.79       179
   macro avg       0.78      0.77      0.77       179
weighted avg       0.79      0.79      0.79       179



In [247]:
errors = X_test.copy()

errors["Actual"] = y_test
errors["Predicted"] = y_pred

errors = errors[errors["Actual"] != errors["Predicted"]]

print("Number of errors:", len(errors))

display(
    errors[
        [
            "Pclass",
            "Sex",
            "AgeGroup",
            "Fare",
            "Embarked",
            "FamilySize",
            "isAlone",
            "title",
            "Actual",
            "Predicted"
        ]
    ].sort_values(["Actual", "Sex", "Pclass"])
)

Number of errors: 38


,Pclass,Sex,AgeGroup,Fare,Embarked,FamilySize,isAlone,title,Actual,Predicted
297,1,female,Child,151.5500,S,4,0,Miss,0,1
199,2,female,YoungAdult,13.0000,S,1,1,Miss,0,1
502,3,female,NaN,7.6292,Q,1,1,Miss,0,1
578,3,female,NaN,14.4583,C,2,0,Mrs,0,1
38,3,female,Teen,18.0000,S,3,0,Miss,0,1
593,3,female,NaN,7.7500,Q,3,0,Miss,0,1
501,3,female,YoungAdult,7.7500,Q,1,1,Miss,0,1
536,1,male,Adult,26.5500,S,1,1,Rare,0,1
698,1,male,Adult,110.8833,C,3,0,Mr,0,1
745,1,male,Senior,71.0000,S,3,0,Rare,0,1


In [248]:
print("Errors by Sex:")
print(errors["Sex"].value_counts())

print("\nErrors by Pclass:")
print(errors["Pclass"].value_counts())

print("\nErrors by Title:")
print(errors["title"].value_counts())

Errors by Sex:
Sex
male      25
female    13
Name: count, dtype: int64

Errors by Pclass:
Pclass
3    18
1    17
2     3
Name: count, dtype: int64

Errors by Title:
title
Mr      22
Miss     9
Mrs      4
Rare     3
Name: count, dtype: int64


In [249]:
false_positives = errors[
    (errors["Actual"] == 0) &
    (errors["Predicted"] == 1)
]

false_negatives = errors[
    (errors["Actual"] == 1) &
    (errors["Predicted"] == 0)
]

print("False Positives:", len(false_positives))
print("False Negatives:", len(false_negatives))

print("\nFalse Positives:")
display(
    false_positives[
        [
            "Pclass",
            "Sex",
            "AgeGroup",
            "Fare",
            "Embarked",
            "FamilySize",
            "isAlone",
            "title",
            "Actual",
            "Predicted"
        ]
    ]
)

print("\nFalse Negatives:")
display(
    false_negatives[
        [
            "Pclass",
            "Sex",
            "AgeGroup",
            "Fare",
            "Embarked",
            "FamilySize",
            "isAlone",
            "title",
            "Actual",
            "Predicted"
        ]
    ]
)

False Positives: 18
False Negatives: 20

False Positives:


,Pclass,Sex,AgeGroup,Fare,Embarked,FamilySize,isAlone,title,Actual,Predicted
536,1,male,Adult,26.5500,S,1,1,Rare,0,1
698,1,male,Adult,110.8833,C,3,0,Mr,0,1
745,1,male,Senior,71.0000,S,3,0,Rare,0,1
297,1,female,Child,151.5500,S,4,0,Miss,0,1
502,3,female,NaN,7.6292,Q,1,1,Miss,0,1
578,3,female,NaN,14.4583,C,2,0,Mrs,0,1
38,3,female,Teen,18.0000,S,3,0,Miss,0,1
593,3,female,NaN,7.7500,Q,3,0,Miss,0,1
442,3,male,YoungAdult,7.7750,S,2,0,Mr,0,1
501,3,female,YoungAdult,7.7500,Q,1,1,Miss,0,1



False Negatives:


,Pclass,Sex,AgeGroup,Fare,Embarked,FamilySize,isAlone,title,Actual,Predicted
553,3,male,YoungAdult,7.2250,C,1,1,Mr,1,0
559,3,female,Adult,17.4000,S,2,0,Mrs,1,0
216,3,female,YoungAdult,7.9250,S,1,1,Miss,1,0
279,3,female,Adult,20.2500,S,3,0,Mrs,1,0
712,1,male,Adult,52.0000,S,2,0,Mr,1,0
455,3,male,YoungAdult,7.8958,C,1,1,Mr,1,0
17,2,male,NaN,13.0000,S,1,1,Mr,1,0
679,1,male,Adult,512.3292,C,2,0,Mr,1,0
431,3,female,NaN,16.1000,S,2,0,Mrs,1,0
55,1,male,NaN,35.5000,S,1,1,Mr,1,0


In [250]:
df["Sex_Pclass_Title"] = (
    df["Sex"] + "_" +
    df["Pclass"].astype(str) + "_" +
    df["title"]
)

In [252]:
# 1. Create the interaction feature
df["Sex_Pclass_Title"] = (
    df["Sex"] + "_" +
    df["Pclass"].astype(str) + "_" +
    df["title"]
)


# 2. Define features
features = [
    "Sex_Pclass_Title",
    "AgeGroup",
    "Fare",
    "Embarked",
    "FamilySize",
    "isAlone"
]

numeric_features = [
    "Fare",
    "FamilySize",
    "isAlone"
]

categorical_features = [
    "Sex_Pclass_Title",
    "AgeGroup",
    "Embarked"
]


# 3. X and y
X_fe = df[features]
y = df["Survived"]


# 4. Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


# 5. Current best XGBoost configuration
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ))
])


# 6. Same 5-fold CV
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# 7. Evaluate
scores = cross_val_score(
    xgb_model,
    X_fe,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Fold scores:", scores)
print("Mean accuracy:", scores.mean())
print("Std:", scores.std())

Fold scores: [0.82681564 0.83146067 0.78089888 0.83146067 0.84831461]
Mean accuracy: 0.8237900947837549
Std: 0.02266247036581904


In [257]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Split the NEW feature set
X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train only on training data
xgb_model.fit(X_train_fe, y_train_fe)

# Predict on unseen test data
y_pred = xgb_model.predict(X_test_fe)

# Accuracy
print("Test Accuracy:", accuracy_score(y_test_fe, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_fe, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test_fe, y_pred))

Test Accuracy: 0.7821229050279329

Confusion Matrix:
[[94 16]
 [23 46]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.85      0.83       110
           1       0.74      0.67      0.70        69

    accuracy                           0.78       179
   macro avg       0.77      0.76      0.77       179
weighted avg       0.78      0.78      0.78       179

